# Notebook 02 - Prepare PTB-XL (thesis Sections 3.4.1-3.4.2)
1. Reads all 100 Hz records (12 leads x 1,000 samples).
2. Filters them: 0.5 Hz high-pass for baseline wander, then the 50 Hz notch (see the note in Step 2.2).
3. Splits by the official patient-level folds: 1-8 train, 9 validation, 10 test.
4. Standardises each lead with the mean and SD of the **training folds only**.
5. Saves the arrays to Drive so later notebooks load them in seconds.

Run notebook 01 first. Time: about 10-15 minutes.

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

In [ ]:
from preprocessing.ptbxl import load_metadata, load_signals, labels_array, SUPERCLASSES, LEADS
from preprocessing.filters import highpass, notch, fit_lead_stats, apply_zscore
from preprocessing.splits import split_by_fold, check_no_patient_overlap
paths = json.load(open(f"{PROJECT}/paths.json"))
PTBXL = paths["PTBXL"]
if IN_COLAB and not os.path.exists(PTBXL):          # new Colab session: unzip again from Drive
    !unzip -q -o "{DATA}/ptb-xl.zip" -d /content/ptb-xl
meta = load_metadata(PTBXL)
print(len(meta), "records")

## Step 2.1 - Read the raw signals (5-10 minutes)

In [ ]:
X_raw = load_signals(PTBXL, meta, sampling_rate=100)
Y = labels_array(meta)
print("X_raw:", X_raw.shape, X_raw.dtype, "  Y:", Y.shape)

**Expected:** `X_raw: (21388, 12, 1000) float32   Y: (21388, 5)` on the real data.

## Step 2.2 - Filter
**Note for your thesis.** At 100 Hz the highest frequency a signal can contain is 50 Hz (the Nyquist frequency), so 50 Hz mains interference was already removed when PTB-XL was downsampled from 500 Hz. A 50 Hz notch filter cannot be built at 100 Hz, so the code skips it with a warning. It is applied automatically wherever the sampling rate is higher (the 500 Hz ablation, and EchoNext before it is resampled). Say this in one sentence in Chapter 4.

In [ ]:
X_f = notch(highpass(X_raw, fs=100, cutoff=0.5), fs=100, freq=50)
i = 0
t = np.arange(1000) / 100
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(t, X_raw[i, 1], color="grey"); ax[0].set_title("Lead II before filtering")
ax[1].plot(t, X_f[i, 1], color="#1f6f8b"); ax[1].set_title("Lead II after 0.5 Hz high-pass")
ax[1].set_xlabel("time (s)"); plt.tight_layout()
plt.savefig(f"{RESULTS}/figures/fig4_2_filtering_example.png", dpi=300); plt.show()

## Step 2.3 - Split by patient-level folds and standardise

In [ ]:
splits = split_by_fold(meta)
check_no_patient_overlap(meta, splits)
stats = fit_lead_stats(X_f[splits["train"]])          # training folds 1-8 only
print("per-lead mean:", stats["mean"].round(3))
print("per-lead SD:  ", stats["std"].round(3))
arrays = {}
for name, idx in splits.items():
    arrays[f"X_{name}"] = apply_zscore(X_f[idx], stats)
    arrays[f"y_{name}"] = Y[idx]
    arrays[f"ecg_id_{name}"] = meta.index.to_numpy()[idx]
    print(f"{name:5s}: X {arrays[f'X_{name}'].shape}  positives per class {Y[idx].sum(0).astype(int)}")
print("Training mean after z-score (should be about 0):", float(arrays["X_train"].mean()).__round__(4))

## Step 2.4 - Save to Drive

In [ ]:
out = f"{DATA}/processed/ptbxl100"
os.makedirs(out, exist_ok=True)
for k, v in arrays.items():
    np.save(f"{out}/{k}.npy", v)
np.savez(f"{out}/lead_stats.npz", **stats)
meta.to_pickle(f"{out}/meta.pkl")
print("Saved to", out, ":", sorted(os.listdir(out)))
del X_raw, X_f

## Step 2.5 - Check the augmentation (used during training only)
Time shift (up to 0.5 s), amplitude scaling (0.9-1.1) and small Gaussian noise; no interpolation between records (no SMOTE), as in Section 3.3.4.

In [ ]:
from training.train import augment
x = torch.as_tensor(arrays["X_train"][:1])
xa = augment(x.repeat(3, 1, 1)).numpy()
plt.figure(figsize=(12, 3))
plt.plot(t, x[0, 1], "k", lw=1.2, label="original")
for j in range(3):
    plt.plot(t, xa[j, 1], lw=0.7, alpha=0.8, label=f"augmented {j+1}")
plt.legend(); plt.title("Lead II, original and augmented"); plt.xlabel("time (s)")
plt.savefig(f"{RESULTS}/figures/augmentation_example.png", dpi=200); plt.show()

**Done.** For Chapter 4 keep Figure 4.2 (filtering) and the per-lead statistics. Next: notebook 03.